<center style="padding: 1em 2em;  margin: 1em 0;">
    <h1 style="font-size: 2.2em; margin: 0 0 0.5em 0; color:rgb(52, 72, 92);">
        «Генераторы, itertools, файлы»
    </h1>
    <div style="font-size: 1.4em; color:rgb(52, 72, 92); font-weight: 600;">
        14 октября 2026
    </div>
</center>

Кадр на паузе мы продолжаем так: `next(steps)` – это `steps.send(None)`.

Так к генератору и обращаются: продолжить кадр и передать значение внутрь. По умолчанию это значение – `None`.

Дальше тот же вызов, только значение уже не `None`. Можно бросить исключение или закрыть кадр.

Потом – готовые обходы, которые мы уже писали руками.

Потом файл: сначала куда уходит `print`. Окажется, что это тот же род объектов, и элемент – строка с диска.

---

## **Сегодня пройдемся по этому плану:**

1. **Кадр снаружи** – значение, исключение, закрытие.
2. **`yield from`** – отдать обход другому.
3. **`itertools`** – тот же протокол, без своего `while`.
4. **Файл и потоки** – куда пишет `print`, несколько строк с клавиатуры, байты картинки.

Каждый кусок нужен следующему.

---

## **Кадр снаружи.**

### **next передаёт None.**

`next(steps)` – это `steps.send(None)`.

Так к генератору и обращаются: продолжить кадр и передать значение внутрь. По умолчанию это значение – `None`.

Покажем это на маленьком генераторе. `next` и `send(None)` оба доводят кадр до первого `yield` и оба отдают наружу `"жду"`. До этой паузы класть значение ещё некуда.

Дальше кадр стоит на `yield`. Следующий `next` – это тоже `send(None)`, и теперь `None` виден: генератор печатает то, что принял.


In [ ]:
def inbox():
    received = yield "жду"
    print("получили", received)
    yield "готово"


by_next = inbox()
print(next(by_next))  # -> жду

by_send = inbox()
print(by_send.send(None))  # -> жду

# Кадр стоит на yield. next передаёт туда None.
print(next(by_next))
# -> получили None
# -> готово

print(by_send.send(None))
# -> получили None
# -> готово


На печати оба вторых вызова пишут «получили None». Значение, которое попало в стоящий `yield`, – `None`.

### **send.**

`yield` отдаёт строку наружу. То же место в теле может значение принять: тогда `yield` стоит в правой части присваивания.

Тот же вызов, только значение уже не `None`: оно попадает в `received = yield ...`.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>send(value):</strong><br>Продолжить кадр и подставить <code>value</code> в тот <code>yield</code>, на котором генератор стоит. Результат <code>send</code> – следующее значение из <code>yield</code>, как у <code>next</code>.
</div>


In [ ]:
def echo():
    received = yield "ready"
    print("получили", received)
    yield received


steps = echo()
print(next(steps))            # -> ready
print(steps.send("привет"))
# -> получили привет
# -> привет

ready
получили привет
привет


`next(steps)` дошёл до `yield "ready"` и отдал строку. До первого `yield` принимать ещё нечему.

`steps.send("привет")` – тот же вызов. `"привет"` становится значением этого `yield` и попадает в `received`. Дальше тело идёт до следующего `yield`.

Вот устройство того, что только сделали: сначала доходим до паузы, потом снаружи кладём значение в тот `yield`, на котором кадр стоит.

![yield-expr](https://i0.wp.com/storage.googleapis.com/ssivart/super9-blog/priming-generator.png?w=1200&ssl=1)

`send` до первого `yield` бросает `TypeError`: кадра на паузе ещё нет.

Для обычного обхода хватает `next` и `for`.

`send` нужен, когда снаружи решают, что кадр увидит после паузы.

Генератор – не отдельная нить выполнения, тело само по себе дальше не идёт: кадр лежит в объекте-генераторе и спит, пока не придёт `next` или `send`.

---

### **throw.**

Та же пауза, другой вход: не значение, а исключение.

Оно возникает *в той строке `yield`, где кадр стоит*.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>throw(ошибка):</strong><br>Возбудить исключение внутри приостановленного кадра, в текущем <code>yield</code>. Если тело его поймало, генератор может отдать следующее значение. Если нет – ошибка выходит наружу, кадр кончается.
</div>


In [17]:
def ticking():
    try:
        yield 1
        yield 2
    except ValueError as error:
        print("поймали внутри", error)
        yield "после ошибки"


steps = ticking()
print(next(steps))  # -> 1
print(steps.throw(ValueError("стоп")))
# -> поймали внутри стоп
# -> после ошибки

1
поймали внутри стоп
после ошибки


После первого `next` кадр стоит на `yield 1`.

`throw` не продолжает тело «как ни в чём не бывало»: в этом `yield` возникает `ValueError`.

`except` его ловит, и следующий `yield` отдаёт другую строку.

Без `except` та же ошибка вышла бы из `throw` наружу, как из обычного вызова.

Одного типа здесь достаточно: важен вход в кадр, не каталог исключений.

---

### **close.**

Мы можем перестать читать генератор в середине.

Кадр при этом жив: в нём могут быть открытые имена, которые нужно отпустить.

Ждать, пока сборщик мусора дойдёт до объекта, неудобно: момент неизвестен.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>close():</strong><br>Попросить генератор закончиться. В текущий <code>yield</code> приходит <code>GeneratorExit</code>. После <code>close</code> следующий <code>next</code> – <code>StopIteration</code>.
</div>

`GeneratorExit` – исключение «этот кадр закрывают».

Его нельзя проглотить и жить дальше: после `except GeneratorExit` нужно выйти.

`finally` – блок после `try`, который выполняется при любом выходе: обычный конец, исключение, `close`. Ресурс отпускают здесь.

In [18]:
def ticking():
    try:
        yield 1
        yield 2
    finally:
        print("кадр закрываем")


steps = ticking()
print(next(steps))  # -> 1
steps.close()       # -> кадр закрываем
# next(steps)  # StopIteration: кадр уже закрыт

1
кадр закрываем


До второго `yield` мы не дошли.

`finally` всё равно напечатал: закрытие – такой же выход из `try`, как конец тела.

`close` нужен, когда генератор держит ресурс.

Чуть позже тот же жест увидим у файла: вышли – закрыли, даже если чтение оборвалось.

`for` по конечному генератору сам доходит до конца; явный `close` – когда мы ушли раньше.

### **except Exception не ловит.**

`close` подставляет `GeneratorExit` в тот `yield`, где кадр стоит.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>GeneratorExit:</strong><br>Исключение закрытия кадра. Наследник <code>BaseException</code>, не <code>Exception</code>. <code>except Exception</code> его не ловит. Обычный обработчик ошибок закрытие не отменяет.
</div>

In [ ]:
def ticking():
    try:
        yield 1
    except Exception:
        print("поймали Exception")
        yield "после Exception"


steps = ticking()
print(next(steps))  # -> 1
steps.close()
print("кадр закрыт")  # -> кадр закрыт

Строки «поймали Exception» в выводе нет.

`close` при этом вернулся. `GeneratorExit` дошёл до конца кадра и закрыл его.

### **yield после close.**

Поймать `GeneratorExit` и снова дойти до `yield` – уже не следующее значение.

`close` просит остановиться.

In [19]:
def stubborn():
    try:
        yield 1
    except GeneratorExit:
        yield "ещё"  # close просил остановиться


steps = stubborn()
print(next(steps))  # -> 1
try:
    steps.close()
except RuntimeError as error:
    print(type(error).__name__)  # -> RuntimeError
    print(error)  # -> generator ignored GeneratorExit

1
RuntimeError
generator ignored GeneratorExit


Ещё одно значение после закрытия отдать нельзя.

---

### **yield from.**

Два готовых генератора хочется склеить в один обход, не копируя цикл.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>yield from:</strong><br>Отдать получение элементов другому итерируемому. Пока тот не кончится, наружу идут его значения. Потом тело продолжается.
</div>

In [ ]:
def countdown(n):
    while n > 0:
        yield n
        n = n - 1


def two_counts():
    yield from countdown(2)
    yield from countdown(2)


print(list(two_counts()))  # -> [2, 1, 2, 1]

[2, 1, 2, 1]


Это не «список плюс список».

Каждый `yield from` просит следующий элемент у своего генератора и сразу отдаёт его наружу.

Когда первый `countdown` кончился, берётся второй.

`for n in countdown(2): yield n` делает то же самое.

`yield from` эту делегацию записывает в одну строку.

Склеить два итератора можно и снаружи, без своей функции.

Этим сейчас займёмся.

---

## **itertools.**

Свой `while` и свой `yield` мы уже написали.

Часть таких обходов нужна часто: отрезать кусок, склеить два источника, пройти короткий список по кругу.

Их не собирают вручную – берут из `itertools`.

Это не новый протокол.

Те же итераторы, те же `next`.

### **count и islice.**

Бесконечный счётчик мы уже писали.

Готовый – `count`.

Конца у него нет: `list(count())` не остановится.

Отрезать первые несколько значений, не собирая `for` с `break`, умеет `islice`.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>count и islice:</strong><br><code>count(start)</code> – бесконечный итератор целых: <code>start</code>, потом <code>start + 1</code>, и так дальше. <code>islice(iterable, n)</code> – итератор первых <code>n</code> элементов. Исходный при этом продвигается.
</div>

In [41]:
from itertools import count, islice

print(list(islice(count(0), 3)))  # -> [0, 1, 2]

[0, 1, 2]


`islice` здесь – тот же выход, что `break` в `for` по бесконечному счётчику.

Разница: обход уже готов, его можно передать дальше.

У списка есть срез `[2:5]`.

У итератора среза нет: он не знает будущих элементов.

`islice(iterator, 2, 5)` как раз этот вопрос закрывает.

Пропущенные значения он тоже забирает `next` и выбрасывает – исходный итератор сдвигается.

### **chain.**

`yield from` склеивал обходы *внутри* своей функции.

Снаружи та же нужда: два итератора уже есть, хочется один за другим.

`range` и список `+` не складывает.

`chain` идёт по первому источнику, потом по второму.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>chain(a, b, ...):</strong><br>Итератор, который сначала отдаёт элементы <code>a</code>, затем <code>b</code>, и так по аргументам.
</div>

In [42]:
from itertools import chain


def countdown(n):
    while n > 0:
        yield n
        n = n - 1


print(list(chain(countdown(2), ["Анна", "Борис"])))
# -> [2, 1, 'Анна', 'Борис']

[2, 1, 'Анна', 'Борис']


Каждый аргумент обходится своим `iter`.

Общего списка заранее нет.

### **cycle.**

Итератор после конца пустой.

Иногда нужен повтор: два имени, а `next` просят дальше.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>cycle(iterable):</strong><br>Бесконечный итератор: дошёл до конца – начал сначала. Для этого он запоминает уже отданные элементы.
</div>

In [43]:
from itertools import cycle, islice

print(list(islice(cycle(["Анна", "Борис"]), 5)))
# -> ['Анна', 'Борис', 'Анна', 'Борис', 'Анна']

['Анна', 'Борис', 'Анна', 'Борис', 'Анна']


Без `islice` `for` по `cycle` не остановится – как по бесконечному счётчику.

### **zip_longest.**

`chain` шёл один за другим.

Иногда два источника нужно вести *рядом*: очередное число и очередное имя.

Обычный `zip` останавливается, как только кончился короткий.

Вера в таком обходе не появится.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>zip_longest(a, b, fillvalue=...):</strong><br>Как <code>zip</code>, но идёт, пока жив самый длинный. На месте уже кончившегося источника ставит <code>fillvalue</code>.
</div>

In [ ]:
from itertools import zip_longest


def countdown(n):
    while n > 0:
        yield n
        n = n - 1


print(list(zip(countdown(2), ["Анна", "Борис", "Вера"])))
# -> [(2, 'Анна'), (1, 'Борис')]

print(list(zip_longest(countdown(2), ["Анна", "Борис", "Вера"], fillvalue=0)))
# -> [(2, 'Анна'), (1, 'Борис'), (0, 'Вера')]

Короткий `countdown` кончился, длинный список – нет. `fillvalue` держит пару на месте, а не выбрасывает хвост.

### **pairwise.**

По `countdown` мы уже идём по одному числу.

Иногда нужна пара соседей: текущее и следующее, без своего индекса.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>pairwise(iterable):</strong><br>Итератор пар <code>(элемент, следующий)</code>. На трёх значениях получится две пары.
</div>

In [ ]:
from itertools import pairwise


def countdown(n):
    while n > 0:
        yield n
        n = n - 1


print(list(pairwise(countdown(3))))
# -> [(3, 2), (2, 1)]

Четвёрки `(3, 2, 1)` здесь нет: один шаг – одна соседняя пара, и итератор при этом продвигается.

### **product.**

Два коротких списка уже есть. Хочется не склеить их и не идти рядом, а получить *каждую* пару: каждое имя с каждым временем.

Это один вопрос про все пары, не три разные функции.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>product(a, b):</strong><br>Итератор всех пар: каждый элемент <code>a</code> с каждым элементом <code>b</code>.
</div>

In [44]:
from itertools import product

names = ["Анна", "Борис"]
times = ["утро", "вечер"]
print(list(product(names, times)))
# -> [('Анна', 'утро'), ('Анна', 'вечер'),
#     ('Борис', 'утро'), ('Борис', 'вечер')]

[('Анна', 'утро'), ('Анна', 'вечер'), ('Борис', 'утро'), ('Борис', 'вечер')]


Четыре пары из двух и двух. Общего списка заранее мы не собирали: `product` отдаёт очередную пару тем же `next`.

### **tee.**

Итератор одноразовый: это мы уже видели у `countdown`.

Иногда один источник нужно пройти дважды, а второго объекта нет.

Обычная копия не поможет: `iter` у итератора вернёт его самого.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>tee(iterable, n):</strong><br><code>n</code> независимых итераторов по тем же элементам. Исходный после этого больше не трогаем.
</div>

In [ ]:
from itertools import tee


def countdown(n):
    while n > 0:
        yield n
        n = n - 1


left, right = tee(countdown(2), 2)
print(list(left))   # -> [2, 1]
print(list(right))  # -> [2, 1]

Два обхода увидели одну и ту же последовательность.

Если после `tee` спросить `next` у исходного `countdown`, копии и оригинал начнут мешать друг другу. Дальше работаем только с тем, что вернул `tee`.

### **accumulate.**

Очередное число из `countdown` мы уже умеем брать.

Иногда нужно не само число, а сумма уже отданных: 3, потом 3 + 2, потом 5 + 1.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>accumulate(iterable):</strong><br>Итератор нарастающих итогов. Первый элемент как есть, каждый следующий – итог вместе с новым.
</div>

In [35]:
from itertools import accumulate


def countdown(n):
    while n > 0:
        yield n
        n = n - 1


print(list(accumulate(countdown(3))))
# -> [3, 5, 6]

[3, 5, 6]


Три шага, три итога. Свой `total = total + n` в этом обходе уже не пишем.

Дальше – файл. Сначала посмотрим, куда уходит `print`. Потом увидим, что тот же объект отдаёт строки через `next`.

---

## **Файл и потоки.**

### **Куда пишет print.**

Мы много раз вызывали `print`, но не смотрели, куда именно уходит текст.

Давайте посмотрим, как тем же `print` писать сразу в файл: у него есть аргумент `file`.

`open` открывает путь на диске и возвращает файловый объект. Что у него общего с консолью – через шаг.

In [45]:
notes = open("../../Other/notes.txt", mode="w", encoding="utf-8")
print("Анна", file=notes)
print("Борис", file=notes)
notes.close()

`close` закрывает файл и перед этим выталкивает буфер. Если между `open` и `close` случится ошибка, выполнение до `close` может не дойти.

`with` сам вызывает этот `close` на выходе из блока: и при обычном конце, и при ошибке внутри. Тот же жест, что `close` у генератора: ресурс отпускаем сами.

In [46]:
with open("../../Other/notes.txt", mode="w", encoding="utf-8") as notes:
    print("Анна", file=notes)
    print("Борис", file=notes)

`"w"` в момент `open` делает файл пустым: старое содержимое пропадает. `"a"` дописывает в конец. `"r"` – чтение, это режим по умолчанию. Если файла нет, `"r"` – ошибка, `"w"` и `"a"` создадут пустой.

Сейчас важно другое: текст ушёл не в консоль.

Прочитаем тот же путь и проверим.

In [ ]:
with open("../../Other/notes.txt", encoding="utf-8") as notes:
    print(notes.read(), end="")
# -> Анна
# -> Борис

И правда записалось.

`read()` забрал весь текст одной строкой. Чем это отличается от `next` – чуть ниже, когда файл окажется итератором.

`encoding="utf-8"` пишем явно, чтобы один и тот же файл читался одинаково на разных машинах.

**Хорошо, но как тогда пишется что-либо в консоль?**

А на самом деле внутри используется очень похожая структура.

`print` всё так же пишет в файл.

Но в какой?

С запуска у программы уже открыты три канала. Они лежат в модуле `sys`.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>Поток:</strong><br>Канал, куда можно писать текст и откуда можно читать. Источник может быть клавиатурой, терминалом, другой программой, не только файлом на диске.
</div>

- `sys.stdout` (стандартный поток вывода)
- `sys.stderr` (стандартный поток ошибок)
- `sys.stdin` (стандартный поток ввода)

In [ ]:
import sys

print("обычный вывод")
print("обычный вывод", file=sys.stdout)
print("что-то пошло не так", file=sys.stderr)

Первые две строки – одно и то же: аргумент `file` по умолчанию как раз `sys.stdout`.

Третья ушла в `stderr`.

Если обычный вывод направят в файл, `stderr` может остаться в терминале: ошибка не потеряется среди данных.

### **Тот же объект.**

`open` возвращает файловый объект.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>open(path):</strong><br>Открыть файл и вернуть файловый объект. Режим по умолчанию <code>"r"</code>, текст.
</div>

Файл на диске, `stdout`, `stderr` и `stdin` – один род объектов.

In [ ]:
import sys

notes = open("../../Other/notes.txt", encoding="utf-8")
print(type(notes))
print(type(sys.stdout))
print(type(sys.stderr))
print(type(sys.stdin))
notes.close()

Точное имя класса сейчас не важно: это текстовый файловый объект.

> В ноутбуке имя у `stdout` может отличаться – не пугайтесь. Род тот же.

Открыли `../../Other/notes.txt` на чтение: Анна и Борис на месте, следующим шагом их заберёт `next`.

Поэтому `print(..., file=notes)` пишет в файл тем же приёмом, что `print(..., file=sys.stdout)` пишет в консоль.

### **Несколько строк.**

`input()` читает из того же `sys.stdin` одну строку и отрезает `\n`.

Несколько строк подряд – не новый объект, а тот же поток до конца.

`read()` у файла мы уже видели: забирает всё, что ещё не прочитано.

У `stdin` то же самое: пока поток не закроют, `read` ждёт следующую строку.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>sys.stdin.read():</strong><br>Прочитать весь стандартный ввод до конца потока. Не одну строку, как <code>input()</code>, а столько, сколько успели набрать.
</div>

В терминале конец ввода – `CTRL + D` (на Windows часто `CTRL + Z`, потом `Enter`). Внутри это `EOF`: поток закрыли, `read` вернул набранное.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #f59e0b; background-color: #fffbf0; color: #92400e;">
<strong>⚠️ Предупреждение:</strong><br>Живой <code>sys.stdin.read()</code> и <code>input()</code> в ячейке Jupyter ждут ввода и не отпускают ядро. Ниже тот же <code>read</code>, только поток уже заполнен.
</div>

In [ ]:
from io import StringIO

# input() – одна строка, до Enter, без \n на конце.
# line = input("имя: ")

# Несколько строк – тот же read, что у sys.stdin.
# В терминале это было бы: text = sys.stdin.read()
stream = StringIO("Анна\nБорис\n")
text = stream.read()
print(repr(text))  # -> 'Анна\nБорис\n'

`input()` здесь не хватает: после первой Анны он уже вернул управление.

`read` держит поток открытым и собирает следующие строки в одну `str`.

### **Файл и next.**

Весь текст одной строкой мы уже забирали через `read`.

Часто нужна следующая строка, а не весь файл сразу.

Файловый объект – итератор. Один `next` читает одну строку.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>Файловый объект:</strong><br>Итератор строк. Один <code>next</code> читает одну строку, вместе с <code>\n</code> на конце, если он в файле есть.
</div>

In [ ]:
with open("../../Other/notes.txt", encoding="utf-8") as notes:
    line = next(notes)
    print(repr(line))  # -> 'Анна\n'

`repr` показывает строку так, чтобы был виден `\n`.

`\n` – часть элемента, не добавка `print`.

Поэтому ниже у `print` стоит `end=""`.

Обычный `print(line)` поставит перевод строки второй раз, и между именами появится пустая строка.

Дальше тот же `for`: он забирает следующие строки тем же `next`.

Анна уже забрана.

In [ ]:
with open("../../Other/notes.txt", encoding="utf-8") as notes:
    print(next(notes), end="")  # -> Анна
    for line in notes:
        print(line, end="")     # -> Борис
    for line in notes:
        print("пусто", line)    # тело не выполняется

Файловый объект сам итератор.

`iter(notes)` вернёт сам `notes`, позиция не сбросится. Второй `for` пустой.

Для нового прохода нужен новый `open` – или вернуть позицию. Почему она не сбросилась – следующий шаг.

### **Указатель.**

После `next` Анна уже забрана, и второй `for` молчит.

Это не «файл кончился навсегда».

Это указатель: место, с которого будет следующее чтение.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>Указатель файла:</strong><br>Позиция, с которой пойдёт следующее чтение или запись. После каждой операции он сдвигается вперёд.
</div>

`next` забрал строку – указатель встал за ней.

Дочитали до конца – указатель в конце, как у исчерпанного итератора.

`tell()` спрашивает текущую позицию, пока мы ещё не начали обход.

После `next` текстовый файл держит внутренний буфер, и тот же `tell()` бросает `OSError`. Это не «файл сломался»: итератор уже читает вперёд.

`seek(0)` всё равно возвращает указатель в начало. Для текста надёжное простое движение – именно в начало, не «на N-й символ».

In [ ]:
with open("../../Other/notes.txt", encoding="utf-8") as notes:
    print(notes.tell())  # -> 0
    print(next(notes), end="")  # -> Анна
    # notes.tell()  # OSError: telling position disabled by next()
    for line in notes:
        print(line, end="")  # -> Борис
    for line in notes:
        print("пусто", line)  # тело не выполняется
    notes.seek(0)
    print(next(notes), end="")  # -> Анна

`seek(0)` – способ файла начать сначала, не создавая новый объект.

Новый `open` тоже ставит указатель в начало.

У текста произвольный `seek` на «символ номер N» ненадёжен: байты и символы не совпадают один к одному. К байтам вернёмся чуть ниже.

### **read.**

Одна строка через итератор – то, ради чего мы идём по файлу.

`read` делает другую работу: всё, что ещё не прочитано, становится одной строкой в памяти.

Указатель после этого – в конце.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>read():</strong><br>Прочитать остаток в одну строку <code>str</code>. Для большого файла это как раз сбор всей последовательности сразу.
</div>

In [ ]:
with open("../../Other/notes.txt", encoding="utf-8") as notes:
    print(next(notes), end="")  # -> Анна
    rest = notes.read()
    print(rest, end="")
# Борис – это уже одна строка rest, не второй шаг итератора

`rest` – остаток одним куском, не два шага `next`.

После `read()` спрашивать `next` нечего: указатель уже в конце.

Если вызвать `read()` до цикла, в памяти окажется весь текст, и `for` по файлу ничего не найдёт.

### **Буфер.**

`print` не обязан сразу показать текст на диске или в терминале.

Сначала запись часто садится в память программы.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>Буфер:</strong><br>Кусок памяти, куда сначала попадает запись. На диск или в терминал она уходит не сразу.
</div>

`flush()` выталкивает буфер: то, что ещё в памяти, отправить в поток.

`close` тоже выталкивает буфер, потом закрывает поток.

`print(..., flush=True)` не ждёт, пока буфер наполнится.

Зачем байты сидят в памяти и где без `flush` запись не видна – сразу после системного вызова.

### **Системный вызов.**

Программа сама экран не рисует и сама байты на диск не кладёт.

Она просит операционную систему. Такой запрос – системный вызов.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>write(fd, bytes, n):</strong><br>Системный вызов записи. <code>fd</code> – небольшое целое, номер уже открытого канала. <code>bytes</code> – байты, <code>n</code> – сколько их отдать.
</div>

С запуска заняты три номера.

- `0` – `stdin`
- `1` – `stdout`
- `2` – `stderr`

Запись Python в `stdout` в конце становится `write(1, ...)`.

Дальше решает ядро: терминал, файл или канал к другой программе.

Ядро – часть операционной системы, которая выполняет системный вызов.

In [ ]:
notes = open("../../Other/notes.txt", mode="a", encoding="utf-8")
fd = notes.fileno()
print(fd)
print(fd > 2)  # -> True
notes.close()

Номер небольшой. `0`, `1` и `2` уже заняты `stdin`, `stdout` и `stderr`, файл получил следующий номер.

В ноутбуке у `stdout` номер может быть другим: вывод уже перенаправили. В обычном запуске `python` запись на экран идёт через `1`.

### **Зачем копить.**

Каждый `write()` в операционную систему дорогой, поэтому байты сидят в памяти: около `8192` или до перевода строки. Один `write` уносит кусок разом.

Это тот же буфер. `flush` его выталкивает.

Так делают, когда в терминале нужен прогресс, когда строку лога надо увидеть сейчас, и когда файл могут не закрыть: до `close` программа может оборваться.

До системы текст доходит так: `TextIOWrapper` → `BufferedWriter` → `write()`.

In [ ]:
import io

print(io.DEFAULT_BUFFER_SIZE)  # -> 8192

out = open("../../Other/pending.txt", mode="w", encoding="utf-8")
print("ещё в памяти", file=out)

viewer = open("../../Other/pending.txt", encoding="utf-8")
print(repr(viewer.read()))  # -> ''

out.flush()
viewer.seek(0)
print(repr(viewer.read()))  # -> 'ещё в памяти\n'

viewer.close()
out.close()

До `flush` чтение из другого `open` пустое: байты сидели в памяти программы. После `flush` строка уже есть.

В терминале нужен тот же `flush`, чтобы прогресс появлялся сразу, а не одной пачкой в конце.

`end=""` говорит `print` не добавлять перевод строки, поэтому точки остаются на той же строке.

In [47]:
import time

print("грузим", end=" ", flush=True)
for _ in range(5):
    time.sleep(0.3)
    print(".", end="", flush=True)
print(" готово")

грузим ..... готово


`\r` возвращает печать в начало строки, поэтому следующее число пишется на то же место.

In [48]:
import time

for i in range(1, 6):
    print(f"\rгрузим {i}/5", end="", flush=True)
    time.sleep(0.3)
print("\rготово   ")

готово   


В ноутбуке числовой индикатор может напечатать несколько строк. В терминале `\r` переписывает одну.

`python -u` и переменная `PYTHONUNBUFFERED` выключают буфер `stdout` у всего процесса. `os.fsync` после `flush` просит систему записать байты на диск.

### **Запись, потом чтение.**

`write` передаёт в файл ровно переданную строку.

`\n` в неё ставим мы.

`print(..., file=notes)` пишет в тот же объект и сам ставит `\n`.

In [ ]:
with open("../../Other/notes.txt", mode="w", encoding="utf-8") as notes:
    notes.write("Анна\n")
    print("Борис", file=notes)

Теперь тот же путь на чтение.

In [ ]:
with open("../../Other/notes.txt", encoding="utf-8") as notes:
    for line in notes:
        print(line, end="")
# -> Анна
# -> Борис

Указатель в начале, `for` забирает строки тем же `next`.

### **Байты.**

Текст, который мы читали, – уже символы: байты файла собраны в `str`.

Картинка, видео и аудио – не текст. Это байты. Читают и пишут их в режиме `"rb"` / `"wb"`, без `encoding`.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>bytes:</strong><br>Последовательность байтов. Текстовый <code>read()</code> возвращает <code>str</code>. В режиме <code>"rb"</code> – <code>bytes</code>.
</div>

Картинку не печатаем целиком. Смотрим длину и первые байты: у JPEG это `ff d8`.

In [49]:
path = "../../Other/images/img.jpg"

with open(path, mode="rb") as picture:
    data = picture.read()

print(data[:2].hex())  # -> ffd8
print(len(data))

ffd8
182505


`data[:2]` – первые два байта, не весь файл. `"wb"` пишет такие же байты обратно.

---

## **Дополнительно почитать.**

Не обязательно до следующей пары.

- [Выражения `yield`](https://docs.python.org/3/reference/expressions.html#yield-expressions) – пауза кадра, `send` / `throw` / `close`, и почему `return` в генераторе становится `StopIteration`.
- [`itertools`](https://docs.python.org/3/library/itertools.html) – те же итераторы: сегодняшние `count`, `islice`, `chain`, `cycle`, `zip_longest`, `pairwise`, `product`, `tee`, `accumulate` и соседние случаи, которых не было.
- [Чтение и запись файлов](https://docs.python.org/3/tutorial/inputoutput.html#reading-and-writing-files) – `open`, обход по строкам и `with` в учебнике языка.
- [`sys.stdin` / `stdout` / `stderr`](https://docs.python.org/3/library/sys.html#sys.stdin) – три стандартных потока, в которые пишет `print`.

На следующей паре будем осматривать данные, которые уже собраны в массив или таблицу: `NumPy` и `Pandas` как инструмент осмотра, не как отдельный курс.